# Lesson 1: Simple ReAct Agent from Scratch (简单的ReAct代理从头开始)

## 本节知识点概括

这一节的目标是从零手写一个最简单的 `ReAct Agent`，帮助你先理解 Agent 的工作原理，再去学 LangGraph 这种更高层的框架。

你会在这个 notebook 里掌握这些核心点：

- `ReAct` 模式的基本循环：`思考 -> 选择工具 -> 观察结果 -> 继续推理 -> 给出最终答案`
- 为什么 Agent 不是“直接回答”，而是会把大任务拆成多个中间步骤
- 工具调用在底层其实就是：模型先输出一个可解析的动作，再由 Python 代码执行工具
- 手写 Agent 的价值：你能看清楚 prompt、解析、工具执行、状态更新分别发生在哪里
- 它和后面 LangGraph 的关系：这一节学的是 Agent 的最小运行机制，后面 LangGraph 学的是如何把这种机制组织成可维护的图工作流

学完这一节，你应该能回答两个问题：
1. 一个 Agent 为什么能“自己决定下一步做什么”？
2. 不靠框架时，Agent 的控制循环最少需要哪些组成部分？


In [ ]:
# based on https://til.simonwillison.net/llms/python-react-pattern

In [1]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

In [2]:
client =  OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),  # 从环境变量读取
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

In [3]:
chat_completion = client.chat.completions.create(
    model="qwen-max",
    messages=[{"role": "user", "content": "Hello world"}]
)

In [4]:
chat_completion.choices[0].message.content

"Hello! It's nice to meet you. How can I assist you today?"

In [5]:
# 定义一个 Agent 类，用于封装大模型的对话逻辑和状态管理
class Agent:
    # 构造函数：在创建 Agent 实例时触发
    def __init__(self, system=""):
        # 将传入的系统提示词（System Prompt）保存在属性中，用于定义 Agent 的角色和行为准则
        self.system = system
        # 初始化消息列表，这是 Agent 的“短期记忆”，用于存储整场对话的历史记录
        self.messages = []
        # 如果提供了系统提示词，则将其作为第一条消息存入历史记录中
        if self.system:
            self.messages.append({"role": "system", "content": system})

    # 特殊方法：让类实例可以像函数一样被直接调用（例如：agent("你好")）
    def __call__(self, message):
        # 将用户的最新输入添加到消息历史中，角色标记为 "user"
        self.messages.append({"role": "user", "content": message})
        # 调用内部的 execute 方法，向大模型发起真正的 API 请求
        result = self.execute()
        # 将大模型返回的回复也存入消息历史中，角色标记为 "assistant"，以维持上下文连贯
        self.messages.append({"role": "assistant", "content": result})
        # 返回大模型的回复文本给调用者
        return result

    # 核心执行方法：负责与底层大模型（LLM）进行通信
    def execute(self):
        # 使用客户端对象（通常是 OpenAI 兼容的 SDK）创建聊天补全请求
        completion = client.chat.completions.create(
                        # 指定使用的模型名称，这里使用的是通义千问的顶级模型 qwen-max
                        model="qwen-max", 
                        # 设置采样温度为 0，这能确保输出结果的高度确定性和稳定性（适合逻辑推导任务）
                        temperature=0,
                        # 将完整的对话历史（包含系统指令、之前的问答和本次提问）发送给模型
                        messages=self.messages)
        # 从模型返回的复杂对象中提取出纯文本内容并返回
        return completion.choices[0].message.content
    

In [6]:
# 提示词中文翻译如下
# 你陷入了思考、行动、暂停、观察的循环中。
# 循环结束时，输出答案。
# 用“思考”一词来描述你对所提问题的想法。
# 使用操作来运行可用的操作之一 - 然后返回 PAUSE。
# 观察结果将是执行这些操作的结果。

# 您可以采取以下行动：

# calculate:
# 例如，计算：4 * 7 / 3
# 运行计算并返回结果——使用 Python，因此必要时请确保使用浮点语法。

# average_dog_weight:
# 例如：average_dog_weight: Collie
# 返回给定品种的狗的平均体重

# 示例会话：

# 问：斗牛犬有多重？
# 想法：我应该用 average_dog_weight 来查看狗狗的体重。
# 操作：平均狗体重：斗牛犬
# 暂停

# 您将再次接到以下电话：

# 观察：一只斗牛犬重 51 磅。

# 然后输出：

# 答案：斗牛犬重 51 磅。
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [8]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [9]:
# 实例化 Agent 类，创建一个名为 abot 的智能代理对象（实例）
# prompt: 这里传入的变量作为 system 参数，确立了该代理的“初始指令”或“人设”
# 此时，构造函数 __init__ 会被调用，将这个 prompt 存入消息历史中
abot = Agent(prompt)

In [10]:
result = abot("How much does a toy poodle weigh?")
print(result)

Thought: I should look up the average weight of a toy poodle using the action `average_dog_weight`.
Action: average_dog_weight: Toy Poodle
PAUSE


In [11]:
result = average_dog_weight("Toy Poodle")

In [12]:
result

'a toy poodles average weight is 7 lbs'

In [13]:
next_prompt = "Observation: {}".format(result)

In [14]:
abot(next_prompt)

"Answer: A toy poodle's average weight is 7 lbs."

In [15]:
abot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user', 'content': 'How much does a 

In [16]:
abot = Agent(prompt)

In [17]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

'Thought: To find the combined weight of the two dogs, I need to first determine the average weight of each breed and then add them together. I will start by using the `average_dog_weight` action for the Border Collie.\nAction: average_dog_weight: Border Collie\nPAUSE'

In [18]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

Observation: a Border Collies average weight is 37 lbs


In [19]:
abot(next_prompt)

'Thought: Now that I have the average weight for the Border Collie, I need to find the average weight for the Scottish Terrier. I will use the `average_dog_weight` action again.\nAction: average_dog_weight: Scottish Terrier\nPAUSE'

In [20]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

Observation: Scottish Terriers average 20 lbs


In [21]:
abot(next_prompt)

'Thought: I now have the average weights for both the Border Collie (37 lbs) and the Scottish Terrier (20 lbs). To find the combined weight, I will add these two values together.\nAction: calculate: 37 + 20\nPAUSE'

In [22]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

Observation: 57


In [23]:
abot(next_prompt)

'Answer: The combined weight of a Border Collie and a Scottish Terrier is 57 lbs.'

### Add loop (让 Agent 能够“反复思考、反复尝试”的架构模式)

In [24]:
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action (选择动作的Python正则表达式)

In [26]:
def query(question, max_turns=5):
    """
    这个函数接收一个'question'（你的初始问题），
    然后让一个 AI 代理循环'max_turns'次（默认5次）来尝试解答它。
    """
    # i 是一个计数器，用来追踪我们进行了多少轮“思考->行动”循环。
    i = 0
    
    # 根据一个预设的'prompt'（指示）来初始化我们的 AI 代理（bot）。
    # 这就像给了 AI 一份“说明书”，告诉它该怎么做。
    bot = Agent(prompt)
    # 'next_prompt' 存储着下一次要发送给 AI 的内容。
    # 一开始，这个内容就是用户最初提的 'question'。
    next_prompt = question
    # 开始主循环。这个循环会一直运行，直到 i 达到 max_turns (5次)。
    # 这是一个“安全阀”，防止 AI 无限循环下去。
    while i < max_turns:
        i += 1

        # --- 思考 (Think) ---
        # 获取AI返回的内容
        result = bot(next_prompt)
        print(result)

        # --- 寻找行动 (Parse Action) ---
        actions = [
            # 1. result.split('\n')：把 AI 的响应按行（\n）分割成一个列表。
            # 2. for a in ...：遍历这个列表中的每一行。
            # 3. if action_re.match(a)：检查这一行是否符合'action_re'（动作正则）定义的格式
            # 4. action_re.match(a)：如果符合，就把这个“匹配对象”存入'actions'列表。
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]

        # --- 检查是否需要行动 ---
        if actions:
            # --- 执行行动 (Act) ---
            # 去除匹配到的行动名称和行动输入
            action, action_input = actions[0].groups()
            # 检查 AI 想要的动作是否在我们预定义的'known_actions'（已知工具）中。
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            # 执行工具！
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            # 如果 'actions' 列表是空的，说明 AI 的响应中没有包含任何“Action”。
            # 这意味着 AI 认为它已经完成了任务，'result'里就是最终答案。
            return

In [27]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

Thought: I need to find the average weight of a border collie and a scottish terrier, then add them together to get their combined weight.
Action: average_dog_weight: Border Collie
PAUSE
 -- running average_dog_weight Border Collie
Observation: a Border Collies average weight is 37 lbs
Thought: Now that I have the average weight of a Border Collie, I need to find the average weight of a Scottish Terrier.
Action: average_dog_weight: Scottish Terrier
PAUSE
 -- running average_dog_weight Scottish Terrier
Observation: Scottish Terriers average 20 lbs
Thought: Now that I have the average weights of both breeds, I can add them together to find their combined weight.
Action: calculate: 37 + 20
PAUSE
 -- running calculate 37 + 20
Observation: 57
Answer: The combined weight of a Border Collie and a Scottish Terrier is 57 lbs.
